# A workflow DAG is a mixture of experts with a fixed router

*ai · advanced* · [open in Tensorizer](https://jackyko1991.github.io/tensorizer/#/ai/moe_dag_pipeline)

An MNIST digit $d$ and a three-word text such as "now plus three" give the answer $y = (d \circ k) \bmod 10$; neither modality alone suffices. Written as a Snakemake-style workflow (encode_image, encode_text, one fuse rule per operation chosen by an input function, classify) and run by the tensor DAG engine of the Circle of Willis problem, the pipeline computes exactly what a top-1 mixture-of-experts layer computes when its router matrix $G$ is fixed to the rule choice. Then the four stages that one tensor program allows: a learned router, experts trained apart and chained, one backward pass through the whole DAG, and distillation into one small dense network stored in int8. A framing that connects known results, not a new theorem.

**Vector form**

$$A_{\mathrm{classify}\leftarrow\mathrm{fuse}} = (I_N \otimes \mathbb 1_E^{\top}) \odot \mathbb 1_N \operatorname{vec}(G)^{\top},\qquad \mathbf o_n = \sum_e G_{ne}\, f_e(\mathbf u_n),\qquad \mathcal L_{\mathrm{aux}} = E \sum_e f_e P_e,\qquad \mathcal L_{\mathrm{KD}} = T^2\, \mathrm{KL}\big(\sigma(\mathbf z^t/T)\,\|\,\sigma(\mathbf z^s/T)\big)$$

**Index (tensor) form**

$$D_{mnec} = G_{mne}\,[c = \mathrm{rank}_e(n)],\qquad U_{meci} = D_{mnec}\, u_{mni},\qquad Y_{mech} = f_e(U_{mec})_h,\qquad o_{mnh} = D_{mnec}\, g_{mne}\, Y_{mech},\qquad g = G\ \text{(workflow)}\ \text{or}\ G \odot p\ \text{(Switch)},\qquad \hat W_{io} = s_o \operatorname{round}(W_{io}/s_o)$$

| Symbol | Code | Shape | Meaning |
|---|---|---|---|
| $A_{\mathrm{rule}}$ | `build_dag(snakefile_text, N)` | (R, R) | rule graph parsed from data/Snakefile with the parser of medical/cow_workflow_dag, plus the input function `routed` (a conditional edge classify ← fuse) |
| $M^{sr}$ | `block(dag, s, r, G)` | (n_s, n_r) | job-level block: the Kronecker product of cow_workflow_dag ($I$, $\mathbb 1$, $\mathbb 1^\top$ per wildcard axis sample, op); for the conditional edge it is masked by the router, so job $n$ of classify reads job $(n, e)$ of fuse iff $G_{ne} = 1$ |
| $G_{ne}$ | `router_matrix(tok)` | (N, E) | the workflow's router: one-hot of the operation word, found by a rule (the input function), not learned |
| $\mathrm{need}_r$ | `needed_jobs(dag, G)` | (n_r,) | jobs the target depends on, $\mathrm{need}_r = \bigvee_s M^{sr\top}\mathrm{need}_s$: only $N$ of the $N E$ fuse jobs |
| $z_{mnv}$ | `run_dag(w, x, tok, dag)` | (M, N, 10) | the engine: rules level by level, each ONE batched call over its needed jobs (fuse: one call per value of its $op$ wildcard) |
| $D_{mnec}$ | `dispatch(G, C)` | (M, N, E, C) | sample $n$ to slot $c$ of expert $e$ (its rank among the samples routed there), as in GShard/Switch; capacity $C$ = largest group |
| $Y_{mech}$ | `einsum('meci,meih->mech', U, W1) …` | (M, E, C, H) | all experts in one batched product; $m$ is the model (seed) axis, never summed |
| $o_{mnh}$ | `einsum('mnec,mne,mech->mnh', D, g, Y)` | (M, N, H) | combine: $e$ and $c$ summed, one nonzero term per sample, so with $g = G$ it equals the DAG output |
| $p_{mne}$ | `gate_probs(w['gate'], h)` | (M, N, E) | learned router: softmax of a linear gate on the text features; top-1 routing, output scaled by $p$ |
| $\mathcal L_{\mathrm{aux}}$ | `load_balance_loss(p, G)` | (M,) | Switch load balancing: $f_e$ = fraction routed to $e$, $P_e$ = mean gate probability; 1 when balanced and uniform, $E$ when collapsed |
| $\mathcal L_{\mathrm{KD}}$ | `kd_loss(z_s, z_t, T)` | (M,) | Hinton distillation at temperature $T$; the student loss is $0.9\,\mathcal L_{\mathrm{KD}} + 0.1$ cross-entropy on the true answer |
| $\hat W_{io}$ | `fake_quant(W)` | (M, I, O) | symmetric int8 per output channel, $s_o = \max_i /W_{io}//127$; stored as int8 + one float32 scale per channel |

Idea. The framing is Jacky Ka Long Ko's: building the Circle of Willis workflow (medical/cow_workflow_dag) showed that a Snakemake DAG can be one tensor program, and such a program looks like a multimodal mixture of experts. Stated precisely: a workflow DAG is an MoE with a fixed router; an MoE is a DAG whose edges are chosen per input. This is a framing that connects known results, not a new theorem, and this problem builds one small instance of it. Equivalence. data/Snakefile has rules encode_image, encode_text, fuse (one job per sample and per operation, the wildcard $op$), classify and collect; classify reads its input through an input function, `routed`, that names the fuse output of the operation in the sample's text. The engine parses the file with the parser of cow_workflow_dag and builds the job adjacency from its Kronecker blocks; the conditional edge's block is $(I_N \otimes \mathbb 1_E^\top) \odot \mathbb 1_N \operatorname{vec}(G)^\top$, so the router matrix $G$ sits inside the adjacency, and only the $N$ chosen of the $N E$ fuse jobs are needed (a boolean backward pass, as Snakemake prunes jobs). Run level by level, one batched call per rule, it gives the same logits as a Switch-style layer (dispatch tensor $D$ by cumulative ranks, all experts in one batched einsum, combine) with $G$ fixed to the rule choice and gate values 1: bitwise identical in float32 and within $4 \times 10^{-15}$ in float64 on all 1000 test samples × 2 seeds (the stacked and the per-group products sum in a different order), the same prediction everywhere. This is an identity of two ways to write one computation, not an empirical finding. Learned router. A softmax gate on the text features with top-1 routing and the Switch load-balancing loss ($\alpha = 0.01$), encoders frozen, two variants side by side. In front of the workflow's own fuse experts the gate alone rediscovers the rule: after the default 800 steps operation $e$ goes to expert $e$ for 0.85 ± 0.03 of the test samples (accuracy 0.82 against 0.92 with the fixed router), still rising (0.95 and 0.90 after 1500 steps in a trial run). Learned together with fresh experts it does not: the routing matches the operations no better than chance (0.35 with the best expert permutation; 1/3 means no relation), the experts stay mixed and the accuracy stays at 0.40. The fuse experts do not see the text, so a routing that ignores the operation leaves each expert an impossible task; but at the start no expert is better at any operation, so a top-1 gate gets no signal to break the symmetry. Gate noise, a 100× larger gate initialisation, a 500-step dense (soft) warm-up and $\alpha = 0.1$ did not change this in trial runs (not part of the default run). Larger MoEs train their routers from scratch successfully; this toy says only that the fixed rule is not free to rediscover here. Stages. (2) Split training: each expert on its own sub-task, the image expert (an MLP) on digit labels (0.916 test accuracy), the text expert (one einsum attention layer) on the number word (1.000), the fuse experts on the 300-entry table from one-hot inputs (1.000); chained through the DAG with no joint training the pipeline answers 0.924 of the test pairs (higher than the digit accuracy because times 0 or times 5 forgive some wrong digits). (3) End-to-end fine-tuning, one backward pass through all stages: 0.924, no gain; the bottleneck is the image expert, and the joint loss brings no new information about digits (same 3000 images). (4) A dense student (784 + 45 → 64 → 64 → 10, 57 930 weights, 0.53 of the pipeline's 108 766) distilled at $T = 4$ (loss $0.9\,\mathcal L_{\mathrm{KD}}$ + 0.1 cross-entropy): 0.85–0.86, agreeing with its teacher on 0.85–0.87 of the samples and still improving at the end of its 3000 steps; int8 per channel: 0.85–0.86, 58.9 kB instead of 232 kB (435 kB for the pipeline). Times for the 1000 test pairs on one CPU thread: DAG engine 90 ms, the same DAG as one MoE layer 19 ms (the engine also builds its job blocks), student 3.4 ms, int8 student 3.5 ms (computed with dequantised floats, so smaller but not really faster here), and one call per job, as a scheduler would run it without process start-up, 900–1000 ms. What does not follow. Only stages that are tensor programs in one memory space fuse with no copy and pass a gradient: file-based tools (OpenFOAM, VMTK, most of bioimage analysis) do not, and a Snakemake DAG of them stays a DAG of processes. The Circle of Willis speed-up came mostly from the avoided per-job process start-up and file I/O, not from batching. And end-to-end training only helps when a stage can learn something the split training did not give it; here it did not. Related work. Neural Module Networks (Andreas et al. 2016) assemble a network per question from modules chosen by a parser, a DAG chosen per input; sparsely gated MoE (Shazeer et al. 2017) and the Switch Transformer (Fedus et al. 2021, top-1 routing and the load-balancing loss used here); modality experts in LIMoE (Mustafa et al. 2022) and Uni-MoE (Li et al. 2024); model stitching (Lenc & Vedaldi 2015; Bansal et al. 2021) joins separately trained networks through a learned interface, as stage 2 does through probability vectors; knowledge distillation (Hinton et al. 2015); DSPy (Khattab et al. 2023) optimises whole pipelines of language-model calls end to end. Google's Pathways (a 2021 blog post and vision) describes one model that routes each task through the parts it needs. Data. MNIST (LeCun, Cortes & Burges; LeCun et al. 1998), licence CC BY-SA 3.0: 3000 training images (300 per digit) and 1000 test images (100 per digit), the first ones of each digit in file order, bundled as uint8 (0.7 MB, data/mnist_train.npz, data/mnist_test.npz, redistributed under the same licence; data/SOURCES.md has the source, checksums and processing). Texts are generated: one operation word (plus, minus, times), one number word (zero to nine) and one filler (now, then) in a random order; the test texts use a fixed seed. Device and limits. The bundled numbers come from the default run on a CPU (the project's environment has no CUDA build of PyTorch here; at about $10^5$ weights a GPU would not help): about 75–150 s of training and measuring plus the GIF on a busy 6-core machine. All seeds train together on the model axis, each with its own minibatches, and every seed is plotted. A toy with one digit and three operations: it shows the structure, not how large multimodal MoEs behave.


## Setup

On Colab, choose **Runtime → Change runtime type → GPU** to run on a GPU — the code uses it automatically.

In [ ]:
import os
import subprocess
import sys

# Shared helpers (problems/_common) come from the repository; skip the clone if we are already inside it.
if not os.path.isdir("problems"):
    if not os.path.isdir("tensorizer"):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/jackyko1991/tensorizer.git"], check=True)
    os.chdir("tensorizer")
sys.path.insert(0, os.getcwd())
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyyaml", "tqdm"], check=True)

import torch

print("torch", torch.__version__, "· device:", "cuda" if torch.cuda.is_available() else "cpu")


## 1. `main.py` — main loop

In [ ]:
%%writefile problems/ai/moe_dag_pipeline/main.py
"""Main loop: MNIST digit + three-word text → (digit op number) mod 10, as a Snakemake-style workflow DAG run by a
tensor engine; the same computation as a fixed-router MoE layer (identical outputs); a learned router; split
training → end-to-end fine-tuning → distillation into one dense student → int8."""

import time
from pathlib import Path

from problems._common.runtime import cli, fast_mode

from . import plot, solve, tensors


def _pm(x) -> str:
    """'mean ± half-range' over the seeds."""
    return f"{float(x.mean()):.3f} ± {float(x.max() - x.min()) / 2:.3f}"


def run(params: dict, out_dir: Path) -> Path:
    fast = fast_mode()
    out_dir = Path(out_dir)
    cfg = tensors.Config(
        seeds=params["seeds"],
        split_steps=40 if fast else params["split_steps"],
        router_steps=20 if fast else params["router_steps"],
        global_steps=20 if fast else params["global_steps"],
        student_steps=40 if fast else params["student_steps"],
        student_hidden=params["student_hidden"],
        temperature=params["temperature"],
        balance_weight=params["balance_weight"],
        test_size=200 if fast else 1000,
        time_repeats=1 if fast else 5,
        seq_samples=10 if fast else 100,
    )
    t0 = time.time()
    res = solve.run_stages(cfg, n_checkpoints=4 if fast else 12)
    M = cfg.seeds
    print(f"{M} seeds on the model axis, {cfg.test_size} test images × one fixed set of texts, CPU: "
          f"{time.time() - t0:.0f} s")  # fmt: skip

    eqv = res["equivalence"]
    jobs = eqv["jobs"]["fuse"]
    print(f"(1) workflow DAG ≡ top-1 MoE with G fixed to the rule choice: fuse runs {jobs[0]} of {jobs[1]} possible "
          "jobs (the router sits in the job adjacency)")  # fmt: skip
    for dt in ("float32", "float64"):
        e = eqv[dt]
        print(f"    {dt}: max |logit difference| = {e['max_abs_diff']:.1e}, bitwise equal: {e['bitwise_equal']}, "
              f"same prediction {e['same_predictions']}/{e['total']}")  # fmt: skip
    rec, ident, load = res["router_recovery"], res["router_identity"], res["router_load"]
    print("    learned router (softmax gate on the text features + load balancing, top-1, 1 of 3 experts):")
    print(f"      gate alone, in front of the workflow's own fuse experts: accuracy {_pm(res['acc']['router_given'])}, "
          f"routes each operation to its own expert for {_pm(ident[:M])} of the test samples")  # fmt: skip
    print(f"      gate and fresh experts learned together: accuracy {_pm(res['acc']['router'])}, routing matches the "
          f"operations (best expert permutation) {_pm(rec[M:])}; 1/3 would be no relation")  # fmt: skip
    for m in range(M):
        print(
            f"      seed {m + 1}, fresh experts: loads "
            + " ".join(f"{v:.2f}" for v in load[M + m].tolist())
            + ", true operation × expert "
            + str([[round(v, 2) for v in r] for r in res["router_confusion"][M + m].tolist()])
        )
    sub = res["subtask_acc"]
    print(f"(2) split training: image expert {_pm(sub['image'])} (digits), text expert {_pm(sub['text'])} "
          f"(number), fuse experts {_pm(sub['fuse'])} (300-entry table); chained, before any joint training: "
          f"{_pm(res['acc']['split'])}")  # fmt: skip
    print(f"(3) end-to-end fine-tuning through the DAG: {_pm(res['acc']['global'])} (digit accuracy of the image "
          f"expert {_pm(res['digit_acc']['split'])} → {_pm(res['digit_acc']['global'])})")  # fmt: skip
    print(f"(4) distilled student (T = {cfg.temperature}): {_pm(res['acc']['student'])}, agrees with its teacher on "
          f"{_pm(res['agree_teacher'])}; int8: {_pm(res['acc']['int8'])}")  # fmt: skip
    print("    model      params   bytes    ms per 1000 samples (1 thread)")
    for k in ("split", "global", "student", "int8"):
        print(f"    {k:8s} {res['params'][k]:8d} {res['bytes'][k]:8d}    {_pm(res['time'][k] * 1e3)}")
    print(f"    same DAG as one MoE layer: {_pm(res['time']['moe'] * 1e3)} ms; one job per call (B = 1, no process "
          f"start-up): {_pm(res['time']['one_by_one'] * 1e3)} ms")  # fmt: skip

    plot.write_tables(res, out_dir)
    plot.stages_png(res, out_dir / "stages.png")
    return plot.animate(res, out_dir / "moe_dag_pipeline.gif", n_frames=12 if fast else 40)


if __name__ == "__main__":
    cli(run, __file__)


## 2. `equation.py` — physics → tensor translation

In [ ]:
%%writefile problems/ai/moe_dag_pipeline/equation.py
r"""The laws as tensor operations: the stages of the pipeline, the top-1 mixture-of-experts layer, and the losses.

Indices: m = model (seed), n = sample, e = expert (= operation), c = slot in an expert's buffer, l, j = word
position, i, h, k, v = channels. Repeated indices are summed unless they appear in the output.

    image expert     z^img_{mnv} = W2_{mhv} relu(W1_{mih} x_{ni} + b1_{mh}) + b2_{mv}                   (i, h summed)
    text expert      X_{mnlc} = T_{m,t_{nl},c} + P_{mlc},  A_{mnlj} = softmax_j(Q_{mnlk} K_{mnjk}/√d)
                     h_{mnc} = mean_l (X + A V W^O)_{mnlc},   z^txt_{mnv} = h_{mnc} W^N_{mcv} + b^N_{mv}
    features         u_{mn} = [softmax z^img_{mn}, softmax z^txt_{mn}]   (20 numbers)
    router           G_{mne} ∈ {0, 1},  Σ_e G_{mne} = 1      (fixed: the parsed operation; learned: top-1 of p_{mne})
    dispatch         D_{mnec} = G_{mne} [c = rank of n among the samples routed to e]
    experts          U_{meci} = D_{mnec} u_{mni},   Y_{mech} = f_e(U_{mec})     (all experts in one batched product)
    combine          o_{mnh} = D_{mnec} g_{mne} Y_{mech}     g = G (fixed router) or G ⊙ p (Switch)
    classify         z_{mnv} = W^C_{mhv} relu(o_{mnh}) + b^C_{mv}
    load balance     ℒ_aux = E Σ_e f_e P_e,  f_e = ⟨G_{ne}⟩_n,  P_e = ⟨p_{ne}⟩_n          (Fedus et al. 2021)
    distillation     ℒ_KD = T² KL(softmax(z^t/T) ‖ softmax(z^s/T))                        (Hinton et al. 2015)
    int8             Ŵ_{io} = s_o round(W_{io}/s_o),  s_o = max_i |W_{io}| / 127              (per output channel)

With G fixed to the workflow's rule choice and g = G, every sum over e and c has exactly one nonzero term, so the
layer computes what the DAG computes: each sample through the one fuse rule its text selects.
"""

import math

import torch
import torch.nn.functional as F


# ---------------------------------------------------------------- stages
def image_logits(img: dict, x: torch.Tensor) -> torch.Tensor:
    """Image expert, an MLP: x (M, N, 784) pixels (each model its own batch) → digit logits (M, N, 10)."""
    hid = torch.relu(torch.einsum("mni,mih->mnh", x, img["W1"]) + img["b1"][:, None])
    return torch.einsum("mnh,mhv->mnv", hid, img["W2"]) + img["b2"][:, None]


def text_features(txt: dict, tok: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
    """Text expert, one bidirectional attention layer: tokens (M, N, 3) → pooled features h (M, N, d) and number
    logits (M, N, 10). Attention rows are probability distributions over the three words."""
    m = torch.arange(tok.shape[0], device=tok.device)[:, None, None]
    X = txt["emb"][m, tok] + txt["pos"][:, None]  # (M, N, L, d): model m looks words up in its own table
    Q, K, Vv = (torch.einsum("mnlc,mck->mnlk", X, txt[k]) for k in ("Wq", "Wk", "Wv"))
    A = torch.softmax(torch.einsum("mnlk,mnjk->mnlj", Q, K) / math.sqrt(X.shape[-1]), dim=-1)
    X = X + torch.einsum("mnlj,mnjk,mkc->mnlc", A, Vv, txt["Wo"])  # j, k summed
    h = X.mean(2)
    return h, torch.einsum("mnc,mcv->mnv", h, txt["Wn"]) + txt["bn"][:, None]


def features(z_img: torch.Tensor, z_txt: torch.Tensor) -> torch.Tensor:
    """u (M, N, 20): the digit and the number as probability vectors, the interface between the stages."""
    return torch.cat([torch.softmax(z_img, -1), torch.softmax(z_txt, -1)], -1)


def expert(u: torch.Tensor, W1, b1, W2, b2) -> torch.Tensor:
    """ONE fuse expert on its own group: u (M, n_e, 20), W1 (M, 20, H) … → (M, n_e, H)."""
    hid = torch.relu(torch.einsum("mni,mih->mnh", u, W1) + b1[:, None])
    return torch.einsum("mnh,mhk->mnk", hid, W2) + b2[:, None]


def experts_stacked(U: torch.Tensor, fuse: dict) -> torch.Tensor:
    """ALL fuse experts at once on their buffers: U (M, E, C, 20) → Y (M, E, C, H)."""
    hid = torch.relu(torch.einsum("meci,meih->mech", U, fuse["W1"]) + fuse["b1"][:, :, None])
    return torch.einsum("mech,mehk->meck", hid, fuse["W2"]) + fuse["b2"][:, :, None]


def classify(cls: dict, o: torch.Tensor) -> torch.Tensor:
    """Shared head: o (M, N, H) → answer logits (M, N, 10)."""
    return torch.einsum("mnh,mhv->mnv", torch.relu(o), cls["W"]) + cls["b"][:, None]


# ---------------------------------------------------------------- the mixture-of-experts layer
def gate_probs(gate: dict, h: torch.Tensor) -> torch.Tensor:
    """Learned router: p (M, N, E) = softmax(h W^G + b^G) from the text features."""
    return torch.softmax(torch.einsum("mnc,mce->mne", h, gate["W"]) + gate["b"][:, None], -1)


def top1(p: torch.Tensor) -> torch.Tensor:
    """G (…, E): one-hot of the most probable expert of each row (top-1 routing, as in Switch)."""
    return F.one_hot(p.argmax(-1), p.shape[-1]).to(p.dtype)


def dispatch(G: torch.Tensor, capacity: int) -> torch.Tensor:
    """D (M, N, E, C): sample n goes to slot c of expert e, its rank among the samples routed to e (a cumulative
    sum, as in GShard/Switch). Samples beyond the capacity would be dropped; here C ≥ the largest group."""
    rank = (torch.cumsum(G, dim=1) - 1) * G  # (M, N, E)
    slot = F.one_hot(rank.long().clamp(0, capacity - 1), capacity).to(G.dtype)
    return G[..., None] * slot * (rank < capacity)[..., None]


def combine(D: torch.Tensor, g: torch.Tensor, Y: torch.Tensor) -> torch.Tensor:
    """o_{mnh} = D_{mnec} g_{mne} Y_{mech}: each sample takes its expert's output, scaled by its gate value."""
    return torch.einsum("mnec,mne,mech->mnh", D, g, Y)


def load_balance_loss(p: torch.Tensor, G: torch.Tensor) -> torch.Tensor:
    """ℒ_aux = E Σ_e f_e P_e per model (M,): 1 for a uniform router with balanced groups, E when it collapses."""
    return p.shape[-1] * (G.mean(1) * p.mean(1)).sum(-1)


# ---------------------------------------------------------------- losses, student, quantisation
def cross_entropy(z: torch.Tensor, y: torch.Tensor) -> torch.Tensor:
    """Mean −log p of the true class per model: z (M, N, V), y (M, N) → (M,)."""
    return -torch.log_softmax(z, -1).gather(-1, y[..., None])[..., 0].mean(1)


def kd_loss(z_s: torch.Tensor, z_t: torch.Tensor, T: float) -> torch.Tensor:
    """T² KL(softmax(z_t/T) ‖ softmax(z_s/T)) averaged over samples, per model (M,) (Hinton et al. 2015)."""
    log_t, log_s = torch.log_softmax(z_t / T, -1), torch.log_softmax(z_s / T, -1)
    return T * T * (log_t.exp() * (log_t - log_s)).sum(-1).mean(1)


def student_logits(st: dict, x: torch.Tensor, tok: torch.Tensor, n_vocab: int) -> torch.Tensor:
    """Dense student: [x, one-hot words] (M, N, 784 + 3·V) → logits (M, N, 10), no routing, no stages."""
    inp = torch.cat([x, F.one_hot(tok, n_vocab).flatten(2).to(x.dtype)], -1)
    h1 = torch.relu(torch.einsum("mni,mih->mnh", inp, st["W1"]) + st["b1"][:, None])
    h2 = torch.relu(torch.einsum("mnh,mhk->mnk", h1, st["W2"]) + st["b2"][:, None])
    return torch.einsum("mnk,mkv->mnv", h2, st["W3"]) + st["b3"][:, None]


def fake_quant(W: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
    """Symmetric int8 per output channel: W (M, I, O) → Ŵ = s·round(W/s) and s (M, 1, O), s = max|W|/127."""
    s = W.abs().amax(1, keepdim=True).clamp(min=1e-12) / 127
    return (W / s).round().clamp(-127, 127) * s, s


## 3. `tensors.py` — PyTorch tensors

In [ ]:
%%writefile problems/ai/moe_dag_pipeline/tensors.py
"""Build the concrete tensors: the bundled MNIST subset, the three-word texts, the router of the workflow, and the
weights of M models stacked on a leading model axis (one per seed, no nn.Module).

data/mnist_train.npz    3000 training images (300 per digit), data/mnist_test.npz 1000 test images (100 per digit),
                        from MNIST: `images` (N, 28, 28) uint8, `labels` (N,) uint8, `index` (N,) in the original file
                        (LeCun, Cortes & Burges, CC BY-SA 3.0, see data/SOURCES.md)
data/Snakefile          the workflow: encode_image, encode_text → fuse_{op} (one expert per operation) → classify

Task: an image of a digit d and a text of three words in any order, one operation word, one number word k and one
filler ("now plus three", "three then minus") → y = (d op k) mod 10. Neither modality alone gives y.
"""

from dataclasses import dataclass
from pathlib import Path

import numpy as np
import torch

DATA = Path(__file__).parent / "data"
SNAKEFILE = DATA / "Snakefile"

OPS = ("plus", "minus", "times")
NUMBERS = ("zero", "one", "two", "three", "four", "five", "six", "seven", "eight", "nine")
FILLERS = ("now", "then")
VOCAB = list(OPS) + list(NUMBERS) + list(FILLERS)  # token ids: operations 0–2, numbers 3–12, fillers 13–14
E, L, P = len(OPS), 3, 28 * 28  # experts (one per operation), words per text, pixels per image
OP_SYMBOL = ("+", "−", "×")


@dataclass
class Config:
    seeds: int = 2  # models on the model axis m, each with its own weights and minibatches
    split_steps: int = 1000  # stage 2: image, text and fuse experts, each on its own sub-task
    router_steps: int = 800  # stage 1b: fresh experts behind a learned softmax gate
    global_steps: int = 400  # stage 3: end-to-end fine-tuning through the whole DAG
    student_steps: int = 3000  # stage 4: distillation into one dense network
    batch: int = 128
    lr: float = 2e-3
    lr_global: float = 3e-4
    hidden_image: int = 128  # image expert 784 → 128 → 10
    d_text: int = 16  # text expert: embedding width
    hidden_fuse: int = 32  # each fuse expert 20 → 32 → 32
    student_hidden: int = 64  # student (784 + 45) → 64 → 64 → 10
    temperature: float = 4.0
    balance_weight: float = 0.01  # Switch load-balancing coefficient α
    kd_hard: float = 0.1  # weight of the true label in the student loss (1 − kd_hard on the teacher)
    test_size: int = 1000
    time_repeats: int = 5  # timing: median of this many calls
    seq_samples: int = 100  # timing of one-job-per-call: this many samples, scaled to the test size


# ---------------------------------------------------------------- data
def load_mnist() -> dict:
    """The bundled subset: x (N, 784) float32 in [0, 1], y (N,) int64, for 'train' (3000) and 'test' (1000)."""
    out = {}
    for split in ("train", "test"):
        with np.load(DATA / f"mnist_{split}.npz") as z:
            out[f"y_{split}"] = torch.from_numpy(z["labels"].astype(np.int64))
            out[f"x_{split}"] = torch.from_numpy(z["images"].reshape(len(z["images"]), P).astype(np.float32) / 255.0)
    return out


def make_texts(n: int, gen: torch.Generator) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    """n random texts → tokens (n, 3), operation (n,), number (n,). The three words are scattered to a random
    permutation of the positions, so the text expert has to find them."""
    op = torch.randint(E, (n,), generator=gen)
    k = torch.randint(10, (n,), generator=gen)
    filler = torch.randint(len(FILLERS), (n,), generator=gen)
    perm = torch.argsort(torch.rand(n, L, generator=gen), dim=1)  # (n, 3) where each word goes
    words = torch.stack([op, len(OPS) + k, len(OPS) + 10 + filler], 1)
    return torch.zeros(n, L, dtype=torch.long).scatter(1, perm, words), op, k


def parse_op(tok: torch.Tensor) -> torch.Tensor:
    """The workflow's input function: the operation named in each text, by rule (no learning). (n, 3) → (n,)."""
    is_op = tok < E
    return (tok * is_op).sum(1)


def parse_number(tok: torch.Tensor) -> torch.Tensor:
    """The number named in each text, by rule (the ground truth of the text expert's sub-task). (n, 3) → (n,)."""
    is_num = (tok >= E) & (tok < E + 10)
    return ((tok - E) * is_num).sum(1)


def router_matrix(tok: torch.Tensor) -> torch.Tensor:
    """G (n, E): G_ne = 1 when the text of sample n names operation e, so the workflow sends it to fuse_e."""
    return torch.nn.functional.one_hot(parse_op(tok), E).float()


def answer(d: torch.Tensor, op: torch.Tensor, k: torch.Tensor) -> torch.Tensor:
    """y = (d + k, d − k or d · k) mod 10 (never negative: torch's % follows the divisor's sign)."""
    return torch.stack([d + k, d - k, d * k], 1).gather(1, op[:, None])[:, 0] % 10


def words(tok_row) -> str:
    return " ".join(VOCAB[int(t)] for t in tok_row)


# ---------------------------------------------------------------- weights, stacked on the model axis m
def _randn(gen, *shape, scale=1.0):
    return torch.randn(*shape, generator=gen) * scale


def init_experts(M: int, cfg: Config, gen: torch.Generator) -> dict:
    """The fuse experts (one per operation, stacked on e), the classify head and the gate of a learned router."""
    Hf, d = cfg.hidden_fuse, cfg.d_text
    return {
        "fuse": {"W1": _randn(gen, M, E, 20, Hf, scale=(2 / 20) ** 0.5), "b1": torch.zeros(M, E, Hf),
                 "W2": _randn(gen, M, E, Hf, Hf, scale=(1 / Hf) ** 0.5), "b2": torch.zeros(M, E, Hf)},
        "cls": {"W": _randn(gen, M, Hf, 10, scale=(2 / Hf) ** 0.5), "b": torch.zeros(M, 10)},
        "gate": {"W": _randn(gen, M, d, E, scale=0.01), "b": torch.zeros(M, E)},
    }  # fmt: skip


def init_weights(M: int, cfg: Config, gen: torch.Generator) -> dict:
    """All stages of the pipeline for M models: image expert, text expert, fuse experts, classify head, gate."""
    Hi, d = cfg.hidden_image, cfg.d_text
    w = {
        "img": {"W1": _randn(gen, M, P, Hi, scale=(2 / P) ** 0.5), "b1": torch.zeros(M, Hi),
                "W2": _randn(gen, M, Hi, 10, scale=(1 / Hi) ** 0.5), "b2": torch.zeros(M, 10)},
        "txt": {"emb": _randn(gen, M, len(VOCAB), d, scale=0.5), "pos": _randn(gen, M, L, d, scale=0.1),
                **{k: _randn(gen, M, d, d, scale=d**-0.5) for k in ("Wq", "Wk", "Wv", "Wo")},
                "Wn": _randn(gen, M, d, 10, scale=d**-0.5), "bn": torch.zeros(M, 10)},
    }  # fmt: skip
    return {**w, **init_experts(M, cfg, gen)}


def init_student(M: int, cfg: Config, gen: torch.Generator) -> dict:
    """One dense network: [image pixels, one-hot words] (784 + 3·15) → H → H → 10."""
    H, n_in = cfg.student_hidden, P + L * len(VOCAB)
    return {"W1": _randn(gen, M, n_in, H, scale=(2 / n_in) ** 0.5), "b1": torch.zeros(M, H),
            "W2": _randn(gen, M, H, H, scale=(2 / H) ** 0.5), "b2": torch.zeros(M, H),
            "W3": _randn(gen, M, H, 10, scale=(1 / H) ** 0.5), "b3": torch.zeros(M, 10)}  # fmt: skip


def leaves(w: dict) -> list[torch.Tensor]:
    return [v for x in w.values() for v in (leaves(x) if isinstance(x, dict) else [x])]


def tmap(w: dict, f) -> dict:
    return {k: tmap(v, f) if isinstance(v, dict) else f(v) for k, v in w.items()}


def cast(w: dict, dtype: torch.dtype) -> dict:
    return tmap(w, lambda t: t.to(dtype))


def clone(w: dict, grad: bool = False) -> dict:
    return tmap(w, lambda t: t.detach().clone().requires_grad_(grad))


def select(w: dict, m: int) -> dict:
    """Model m alone, keeping a model axis of length 1."""
    return tmap(w, lambda t: t[m : m + 1].detach().clone())


def n_params(w: dict) -> int:
    """Weights per model (the model axis divided out)."""
    return sum(t[0].numel() for t in leaves(w))


## 4. `solve.py` — solver

In [ ]:
%%writefile problems/ai/moe_dag_pipeline/solve.py
"""The workflow DAG engine (Snakefile → routed job DAG → one batched call per rule), the same computation as a
top-1 mixture-of-experts layer, and the four stages: split training, a learned router, end-to-end fine-tuning,
distillation and int8 quantisation.

The DAG algebra (Kronecker job adjacency, nilpotency, boolean closure and levels) is imported from
medical/cow_workflow_dag, whose engine runs a real Snakemake workflow; this file adds the one thing that workflow
did not have, a conditional input (an input function), whose job-level block is the router matrix G.
"""

import itertools
import math
import re
import time
from functools import partial

import torch

from problems._common.progress import steps as progress_steps
from problems.medical.cow_workflow_dag import equation as dag_eq
from problems.medical.cow_workflow_dag.solve import parse_snakefile

from . import equation as eq
from . import tensors as T

AXES = ("sample", "op")  # wildcard axes, in job-numbering order (row-major)


# ---------------------------------------------------------------- the workflow DAG
def build_dag(text: str, n_samples: int) -> dict:
    """Rules of the Snakefile, the rule graph A_rule (A_sr = 1 when rule s reads a pattern rule r writes), each
    rule's wildcard axes, the levels, and the conditional edges: an input function such as

        def routed(wildcards): return f"results/{wildcards.sample}/fused_{OP[wildcards.sample]}.npy"

    reads the pattern results/{sample}/fused_{op}.npy with the wildcard `op` chosen per sample."""
    rules = parse_snakefile(text)
    funcs = dict(re.findall(r'^def (\w+)\(wildcards\):\s*\n\s+return f"([^"]+)"', text, flags=re.M))
    routed = dict(re.findall(r"^rule (\w+):\s*\n\s+input: (\w+)\s*$", text, flags=re.M))
    for r in rules:
        if r["name"] in routed:
            pat = re.sub(r"\{wildcards\.(\w+)\}", r"{\1}", funcs[routed[r["name"]]])
            pat = re.sub(r"\{(\w+)\[wildcards\.\w+\]\}", lambda m: "{" + m.group(1).lower() + "}", pat)
            r["inputs"].append((pat, ()))
            r["routed_by"] = routed[r["name"]]
    wild = lambda p: set(re.findall(r"\{(\w+)\}", p))  # noqa: E731
    axes = []
    for r in rules:
        pats = r["outputs"] or [p for p, _ in r["inputs"]]
        wc = set().union(*map(wild, pats)) - ({w for _, agg in r["inputs"] for w in agg} if not r["outputs"] else set())
        axes.append(tuple(a for a in AXES if a in wc))
    R = len(rules)
    A_rule, conditional = torch.zeros(R, R), {}
    for s, rs in enumerate(rules):
        for pat, _ in rs["inputs"]:
            for r, rr in enumerate(rules):
                if pat in rr["outputs"]:
                    A_rule[s, r] = 1
                    chosen = wild(pat) - set(axes[s]) - {w for p2, agg in rs["inputs"] for w in agg}
                    if chosen and rs.get("routed_by"):
                        conditional[(s, r)] = chosen.pop()
    return {"rules": rules, "names": [r["name"] for r in rules], "A_rule": A_rule, "axes": axes,
            "levels": dag_eq.levels(A_rule), "conditional": conditional,
            "sizes": {"sample": n_samples, "op": T.E}}  # fmt: skip


def block(dag: dict, s: int, r: int, G: torch.Tensor) -> torch.Tensor:
    """Job-level block (n_s, n_r) of the edge r → s: the Kronecker product M^{sr} of cow_workflow_dag, and for a
    conditional edge that product masked by the router, (I_N ⊗ 𝟙ᵀ_E) ⊙ vec(G)ᵀ: job n reads job (n, e) iff G_ne = 1."""
    M = dag_eq.kron_block(dag["axes"][r], dag["axes"][s], dag["sizes"])
    if (s, r) in dag["conditional"]:
        M = M * G.reshape(1, -1).to(M.dtype)
    return M


def job_adjacency_routed(dag: dict, G: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
    """The whole (J, J) job adjacency with the router inside (small N only: J = N·(3 + E) + 1)."""
    A, off = dag_eq.job_adjacency(dag["A_rule"], dag["axes"], dag["sizes"])
    for s, r in dag["conditional"]:
        A[off[s] : off[s + 1], off[r] : off[r + 1]] = block(dag, s, r, G)
    return A, off


def needed_jobs(dag: dict, G: torch.Tensor) -> dict:
    """Which jobs the target needs (what Snakemake schedules): the target rule's jobs, then backwards level by level,
    need_r = ⋁_s M^{sr}ᵀ need_s (boolean). Block by block, so the (J, J) matrix is never built."""
    A, lev, names = dag["A_rule"], dag["levels"], dag["names"]
    need = {}
    for r in sorted(range(len(names)), key=lambda i: -int(lev[i])):
        n_r = math.prod(dag["sizes"][a] for a in dag["axes"][r])
        if A[:, r].sum() == 0:  # nobody reads it: a target
            need[names[r]] = torch.ones(n_r, dtype=torch.bool)
            continue
        v = torch.zeros(n_r)
        for s in A[:, r].nonzero().flatten().tolist():
            v = v + block(dag, s, r, G).T @ need[names[s]].float()
        need[names[r]] = v > 0
    return {n: need[n] for n in names}


def run_dag(w: dict, x: torch.Tensor, tok: torch.Tensor, dag: dict) -> tuple[torch.Tensor, dict]:
    """Run the workflow as tensors: rules level by level, each rule ONE call over all its needed jobs. The router is
    the workflow's own input function (the parsed operation), so `fuse` runs only the N chosen of N·E jobs: one
    batched expert call per value of its `op` wildcard. x (N, 784), tok (N, 3) → logits (M, N, 10), times (s)."""
    M = w["img"]["W1"].shape[0]
    xs, ts = x.expand(M, *x.shape), tok.expand(M, *tok.shape)
    G = T.router_matrix(tok)
    need = needed_jobs(dag, G)
    names = dag["names"]
    fz = w["fuse"]

    def fuse(st):
        jobs = need["fuse"].nonzero().flatten()  # (N,) needed (sample, op) jobs, sample-major
        n, e = jobs // T.E, jobs % T.E
        u = eq.features(st["encode_image"], st["encode_text"][1])
        out = torch.zeros(M, len(jobs), fz["W1"].shape[-1], dtype=u.dtype)
        for k in range(T.E):  # the values of the wildcard {op}: one batched call each
            sel = e == k
            out[:, sel] = eq.expert(u[:, n[sel]], fz["W1"][:, k], fz["b1"][:, k], fz["W2"][:, k], fz["b2"][:, k])
        return jobs, out

    def classify(st):
        jobs, out = st["fuse"]
        src = block(dag, names.index("classify"), names.index("fuse"), G).argmax(1)  # the file `routed` names
        return eq.classify(w["cls"], out[:, torch.searchsorted(jobs, src)])

    funcs = {"encode_image": lambda st: eq.image_logits(w["img"], xs),
             "encode_text": lambda st: eq.text_features(w["txt"], ts),
             "fuse": fuse, "classify": classify, "collect": lambda st: st["classify"]}  # fmt: skip
    store, times = {}, {}
    for k in range(int(dag["levels"].max()) + 1):
        for r in (dag["levels"] == k).nonzero().flatten().tolist():
            t0 = time.perf_counter()
            store[names[r]] = funcs[names[r]](store)
            times[names[r]] = time.perf_counter() - t0
    return store["collect"], times


def run_jobs_one_by_one(w: dict, x: torch.Tensor, tok: torch.Tensor) -> torch.Tensor:
    """The same workflow the way a job scheduler runs it: one call per job (B = 1), N·4 calls (no process start-up
    or file I/O counted). Only for timing; same numbers as run_dag."""
    out = []
    for i in range(len(x)):
        xi, ti = x[i : i + 1].expand(1, 1, -1), tok[i : i + 1].expand(1, 1, -1)
        u = eq.features(eq.image_logits(w["img"], xi), eq.text_features(w["txt"], ti)[1])
        k = int(T.parse_op(tok[i : i + 1]))
        fz = w["fuse"]
        o = eq.expert(u, fz["W1"][:, k], fz["b1"][:, k], fz["W2"][:, k], fz["b2"][:, k])
        out.append(eq.classify(w["cls"], o))
    return torch.cat(out, 1)


# ---------------------------------------------------------------- the same computation as a MoE layer
def fuse_classify(w: dict, u: torch.Tensor, G: torch.Tensor, g: torch.Tensor) -> torch.Tensor:
    """Dispatch → all experts in one batched product → combine → classify. u (M, N, 20), G, g (M, N, E)."""
    D = eq.dispatch(G, int(G.sum(1).max()))  # capacity = the largest group: no sample is dropped
    U = torch.einsum("mnec,mni->meci", D, u)
    return eq.classify(w["cls"], eq.combine(D, g, eq.experts_stacked(U, w["fuse"])))


def moe_forward(w: dict, x: torch.Tensor, tok: torch.Tensor, G: torch.Tensor, combine: torch.Tensor) -> torch.Tensor:
    """The pipeline as one MoE layer with a given 0/1 router G and combine weights g (both (M, N, E))."""
    M = w["img"]["W1"].shape[0]
    xs = x if x.dim() == 3 else x.expand(M, *x.shape)
    ts = tok if tok.dim() == 3 else tok.expand(M, *tok.shape)
    u = eq.features(eq.image_logits(w["img"], xs), eq.text_features(w["txt"], ts)[1])
    return fuse_classify(w, u, G, combine)


def switch_forward(w: dict, x: torch.Tensor, tok: torch.Tensor):
    """Learned router (Switch Transformer): p = softmax gate on the text features, G = top-1(p), g = G ⊙ p.
    Returns logits, p, G."""
    h, z_txt = eq.text_features(w["txt"], tok)
    u = eq.features(eq.image_logits(w["img"], x), z_txt)
    p = eq.gate_probs(w["gate"], h)
    G = eq.top1(p)
    return fuse_classify(w, u, G, G * p), p, G


# ---------------------------------------------------------------- training helpers
def accuracy(z: torch.Tensor, y: torch.Tensor) -> torch.Tensor:
    return (z.argmax(-1) == y).float().mean(-1)


def batch(data: dict, M: int, B: int, gen: torch.Generator):
    """Each model its own minibatch: images (M, B, 784), tokens (M, B, 3), digit, operation, number, answer."""
    idx = torch.randint(len(data["x_train"]), (M, B), generator=gen)
    tok, op, k = T.make_texts(M * B, gen)
    d = data["y_train"][idx]
    op, k = op.reshape(M, B), k.reshape(M, B)
    return (
        data["x_train"][idx],
        tok.reshape(M, B, T.L),
        d,
        op,
        k,
        T.answer(d.flatten(), op.flatten(), k.flatten()).reshape(M, B),
    )


def checkpoints(n_steps: int, n: int) -> set:
    return {round(i * n_steps / n) for i in range(n + 1)}


def table_inputs(dtype=torch.float32):
    """All 300 (digit, operation, number) cases as one-hot features u (300, 20), the router G (300, E), answers."""
    d, op, k = (
        t.flatten() for t in torch.meshgrid(torch.arange(10), torch.arange(T.E), torch.arange(10), indexing="ij")
    )
    u = torch.cat([torch.nn.functional.one_hot(d, 10), torch.nn.functional.one_hot(k, 10)], 1).to(dtype)
    return u, torch.nn.functional.one_hot(op, T.E).to(dtype), T.answer(d, op, k)


def float_bytes(w: dict) -> int:
    return 4 * T.n_params(w)


def int8_bytes(st: dict) -> int:
    """Student stored as int8 weight matrices + one float32 scale per output channel + float32 biases."""
    total = 0
    for k, t in st.items():
        total += t[0].numel() + 4 * t.shape[-1] if k.startswith("W") else 4 * t[0].numel()
    return total


def quantise(st: dict) -> dict:
    return {k: eq.fake_quant(t)[0] if k.startswith("W") else t.clone() for k, t in st.items()}


def time_call(f, repeats: int) -> float:
    """Median wall-clock of f() over repeats (s), on one thread so the numbers compare."""
    n = torch.get_num_threads()
    torch.set_num_threads(1)
    try:
        f()
        ts = []
        for _ in range(repeats):
            t0 = time.perf_counter()
            f()
            ts.append(time.perf_counter() - t0)
    finally:
        torch.set_num_threads(n)
    return sorted(ts)[len(ts) // 2]


# ---------------------------------------------------------------- the four stages
def run_stages(cfg: T.Config, n_checkpoints: int = 20, display: int = 10) -> dict:
    """Train and measure everything. All seeds run together on the model axis M."""
    torch.manual_seed(0)
    M = cfg.seeds
    data = T.load_mnist()
    gen = torch.Generator().manual_seed(1)
    xt, yt = data["x_test"][: cfg.test_size], data["y_test"][: cfg.test_size]
    Nt = len(xt)
    tok_t, op_t, k_t = T.make_texts(Nt, torch.Generator().manual_seed(2026))  # one fixed test set of texts
    y_t = T.answer(yt, op_t, k_t)
    Xt, Tt = xt.expand(M, Nt, T.P), tok_t.expand(M, Nt, T.L)
    G_t = T.router_matrix(tok_t).expand(M, Nt, T.E)
    u_tab, G_tab, y_tab = table_inputs()
    dag = T.SNAKEFILE.read_text(encoding="utf-8")
    dag = build_dag(dag, n_samples=Nt)
    res = {"cfg": cfg, "M": M, "curves": {}, "acc": {}, "subtask_acc": {}}

    def end_acc(w):
        with torch.no_grad():
            return accuracy(moe_forward(w, Xt, Tt, G_t, G_t), y_t)

    # stage 2: split training, each expert on its own sub-task (the losses touch disjoint weights)
    w = T.clone(T.init_weights(M, cfg, gen), grad=True)
    opt = torch.optim.Adam(T.leaves({k: w[k] for k in ("img", "txt", "fuse", "cls")}), lr=cfg.lr)
    cps, curve = checkpoints(cfg.split_steps, n_checkpoints), []
    for t in progress_steps(cfg.split_steps + 1, "stage 2: split training"):
        if t in cps:
            with torch.no_grad():
                Gtab = G_tab.expand(M, -1, -1)
                curve.append((t, accuracy(eq.image_logits(w["img"], Xt), yt.expand(M, Nt)),
                              accuracy(eq.text_features(w["txt"], Tt)[1], k_t.expand(M, Nt)),
                              accuracy(fuse_classify(w, u_tab.expand(M, -1, -1), Gtab, Gtab), y_tab.expand(M, -1)),
                              end_acc(w)))  # fmt: skip
        if t == cfg.split_steps:
            break
        x, tok, d, op, k, y = batch(data, M, cfg.batch, gen)
        dd, oo, kk = (torch.randint(n, (M, cfg.batch), generator=gen) for n in (10, T.E, 10))
        u1 = torch.cat([torch.nn.functional.one_hot(dd, 10), torch.nn.functional.one_hot(kk, 10)], -1).float()
        G1 = torch.nn.functional.one_hot(oo, T.E).float()
        y1 = T.answer(dd.flatten(), oo.flatten(), kk.flatten()).reshape(M, -1)
        loss = (eq.cross_entropy(eq.image_logits(w["img"], x), d)
                + eq.cross_entropy(eq.text_features(w["txt"], tok)[1], k)
                + eq.cross_entropy(fuse_classify(w, u1, G1, G1), y1))  # fmt: skip
        opt.zero_grad()
        loss.sum().backward()
        opt.step()
    res["curves"]["split"] = curve
    _, a_img, a_txt, a_fuse, a_chain = curve[-1]
    res["subtask_acc"] = {"image": a_img, "text": a_txt, "fuse": a_fuse}
    w_split = T.clone(w)

    # the DAG engine on the split-trained pipeline, and the equivalence with the fixed-router MoE layer
    with torch.no_grad():
        z_dag, rule_times = run_dag(w_split, xt, tok_t, dag)
        res["acc"]["split"] = accuracy(z_dag, y_t)
        eqv = {}
        for name, dt in (("float64", torch.float64), ("float32", torch.float32)):
            w64 = T.cast(w_split, dt)
            za, _ = run_dag(w64, xt.to(dt), tok_t, dag)
            Gd = G_t.to(dt)
            zb = moe_forward(w64, xt.to(dt), tok_t, Gd, Gd)
            eqv[name] = {"max_abs_diff": float((za - zb).abs().max()), "bitwise_equal": bool(torch.equal(za, zb)),
                         "same_predictions": int((za.argmax(-1) == zb.argmax(-1)).sum()),
                         "total": za.shape[0] * Nt}  # fmt: skip
        need = needed_jobs(dag, T.router_matrix(tok_t))
        eqv["jobs"] = {n: (int(v.sum()), len(v)) for n, v in need.items()}
        res["equivalence"], res["rule_times"] = eqv, rule_times

    # stage 1b: a learned router (softmax gate + load balancing), encoders frozen, two variants side by side on the
    # model axis: models 0…M−1 keep the split-trained fuse experts (only the gate learns), models M…2M−1 get fresh
    # experts that learn together with the gate
    rep2 = lambda t: torch.cat([t, t])  # noqa: E731
    fresh = T.init_experts(M, cfg, torch.Generator().manual_seed(7))
    wr = {"img": T.tmap(w_split["img"], rep2), "txt": T.tmap(w_split["txt"], rep2),
          "fuse": {k: torch.cat([w_split["fuse"][k], fresh["fuse"][k]]) for k in fresh["fuse"]},
          "cls": {k: torch.cat([w_split["cls"][k], fresh["cls"][k]]) for k in fresh["cls"]},
          "gate": T.init_experts(2 * M, cfg, torch.Generator().manual_seed(8))["gate"]}  # fmt: skip
    wr = T.clone(wr, grad=True)
    kept = T.leaves({k: wr[k] for k in ("fuse", "cls")})
    opt = torch.optim.Adam(kept + T.leaves(wr["gate"]), lr=cfg.lr)
    show = torch.cat([(op_t == e).nonzero().flatten()[:display] for e in range(T.E)])  # display rows, by operation
    X2, T2 = xt.expand(2 * M, Nt, T.P), tok_t.expand(2 * M, Nt, T.L)
    cps, curve, snaps = checkpoints(cfg.router_steps, n_checkpoints), [], []
    for t in progress_steps(cfg.router_steps + 1, "stage 1b: learned router"):
        if t in cps:
            with torch.no_grad():
                z, p, G = switch_forward(wr, X2, T2)
                curve.append((t, accuracy(z, y_t), eq.load_balance_loss(p, G)))
                snaps.append(G[:, show].clone())
        if t == cfg.router_steps:
            break
        x, tok, d, op, k, y = batch(data, 2 * M, cfg.batch, gen)
        z, p, G = switch_forward(wr, x, tok)
        loss = eq.cross_entropy(z, y) + cfg.balance_weight * eq.load_balance_loss(p, G)
        opt.zero_grad()
        loss.sum().backward()
        for q in kept:  # the given experts stay as they are (zero gradient ⇒ Adam leaves them unchanged)
            q.grad[:M] = 0
        opt.step()
    with torch.no_grad():
        z, p, G = switch_forward(wr, X2, T2)
    conf = torch.einsum("ne,mnf->mef", torch.nn.functional.one_hot(op_t, T.E).float(), G) / Nt  # true op × expert
    perms = list(itertools.permutations(range(T.E)))
    matched = torch.stack([conf[:, list(range(T.E)), list(q)].sum(1) for q in perms])  # (perms, 2M)
    best = matched.argmax(0)
    res.update({"router_confusion": conf, "router_recovery": matched.max(0).values,
                "router_identity": matched[0], "router_perm": [perms[int(b)] for b in best], "router_snaps": snaps,
                "show": show, "router_load": G.mean(1)})  # fmt: skip
    res["curves"]["router"] = curve
    acc_r = accuracy(z, y_t)
    res["acc"]["router_given"], res["acc"]["router"] = acc_r[:M], acc_r[M:]

    # stage 3: end-to-end fine-tuning through the whole DAG (fixed router), one backward pass through all stages
    wg = T.clone(w_split, grad=True)
    opt = torch.optim.Adam(T.leaves({k: wg[k] for k in ("img", "txt", "fuse", "cls")}), lr=cfg.lr_global)
    cps, curve = checkpoints(cfg.global_steps, n_checkpoints), []
    for t in progress_steps(cfg.global_steps + 1, "stage 3: global fine-tuning"):
        if t in cps:
            curve.append((t, end_acc(wg)))
        if t == cfg.global_steps:
            break
        x, tok, d, op, k, y = batch(data, M, cfg.batch, gen)
        G = T.router_matrix(tok.reshape(-1, T.L)).reshape(M, -1, T.E)
        loss = eq.cross_entropy(moe_forward(wg, x, tok, G, G), y)
        opt.zero_grad()
        loss.sum().backward()
        opt.step()
    res["curves"]["global"] = curve
    w_glob = T.clone(wg)
    with torch.no_grad():
        z_glob, _ = run_dag(w_glob, xt, tok_t, dag)
        res["acc"]["global"] = accuracy(z_glob, y_t)
        res["digit_acc"] = {"split": a_img, "global": accuracy(eq.image_logits(w_glob["img"], Xt), yt.expand(M, Nt))}

    # stage 4: distil the whole DAG into one dense student (Hinton KD), then int8
    st = T.clone(T.init_student(M, cfg, torch.Generator().manual_seed(11)), grad=True)
    opt = torch.optim.Adam(T.leaves(st), lr=cfg.lr)
    cps, curve = checkpoints(cfg.student_steps, n_checkpoints), []
    student = lambda s, x, tok: eq.student_logits(s, x, tok, len(T.VOCAB))  # noqa: E731
    for t in progress_steps(cfg.student_steps + 1, "stage 4: distillation"):
        if t in cps:
            with torch.no_grad():
                curve.append((t, accuracy(student(st, Xt, Tt), y_t)))
        if t == cfg.student_steps:
            break
        x, tok, d, op, k, y = batch(data, M, 2 * cfg.batch, gen)
        with torch.no_grad():
            G = T.router_matrix(tok.reshape(-1, T.L)).reshape(M, -1, T.E)
            zt = moe_forward(w_glob, x, tok, G, G)
        zs = student(st, x, tok)
        loss = (1 - cfg.kd_hard) * eq.kd_loss(zs, zt, cfg.temperature) + cfg.kd_hard * eq.cross_entropy(zs, y)
        opt.zero_grad()
        loss.sum().backward()
        opt.step()
    res["curves"]["student"] = curve
    st = T.clone(st)
    st_q = quantise(st)
    with torch.no_grad():
        z_st, z_q = student(st, Xt, Tt), student(st_q, Xt, Tt)
    res["acc"]["student"], res["acc"]["int8"] = accuracy(z_st, y_t), accuracy(z_q, y_t)
    res["agree_teacher"] = (z_st.argmax(-1) == z_glob.argmax(-1)).float().mean(-1)

    # size and time (per model, one thread)
    pipe_params = T.n_params({k: w_split[k] for k in ("img", "txt", "fuse", "cls")})
    res["params"] = {"split": pipe_params, "global": pipe_params, "student": T.n_params(st), "int8": T.n_params(st)}
    res["bytes"] = {"split": 4 * pipe_params, "global": 4 * pipe_params, "student": float_bytes(st),
                    "int8": int8_bytes(st)}  # fmt: skip
    timing = {k: [] for k in ("split", "global", "student", "int8", "moe", "one_by_one")}
    n_seq, rep = min(cfg.seq_samples, Nt), cfg.time_repeats
    with torch.no_grad():
        for m in range(M):
            ws, wgm, sm, qm = (T.select(v, m) for v in (w_split, w_glob, st, st_q))
            G1 = T.router_matrix(tok_t)[None]
            timing["split"].append(time_call(partial(run_dag, ws, xt, tok_t, dag), rep))
            timing["global"].append(time_call(partial(run_dag, wgm, xt, tok_t, dag), rep))
            timing["moe"].append(time_call(partial(moe_forward, ws, xt, tok_t, G1, G1), rep))
            timing["student"].append(time_call(partial(student, sm, xt[None], tok_t[None]), rep))
            timing["int8"].append(time_call(partial(student, qm, xt[None], tok_t[None]), rep))
            t1 = time_call(partial(run_jobs_one_by_one, ws, xt[:n_seq], tok_t[:n_seq]), max(1, rep // 2))
            timing["one_by_one"].append(t1 * Nt / n_seq)
    res["time"] = {k: torch.tensor(v) for k, v in timing.items()}

    # display: a few test examples, with the route each one takes
    res["examples"] = {"x": xt, "tok": tok_t, "op": op_t, "k": k_t, "d": yt, "y": y_t,
                       "pred_split": z_dag.argmax(-1), "pred_global": z_glob.argmax(-1),
                       "pred_student": z_st.argmax(-1), "pred_int8": z_q.argmax(-1)}  # fmt: skip
    res["G_fixed"] = T.router_matrix(tok_t)[show]
    return res


## 5. `plot.py` — 2D/3D plot → GIF

In [ ]:
%%writefile problems/ai/moe_dag_pipeline/plot.py
"""Animate the four stages: the workflow DAG with the route of a test example, the router matrix G (fixed by the
workflow vs learned by a softmax gate), the training curves of every stage, and accuracy / size / time per model
with one dot per seed. Also a static figure (router confusion, accuracy vs bytes) and the CSV tables."""

import csv
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from matplotlib.colors import ListedColormap
from matplotlib.patches import FancyBboxPatch

from problems._common.plotting import ACCENT, save_png
from problems.ai.chain_of_thought.plot import save_gif_shared_palette

from . import tensors as T

MODELS = ["split", "global", "student", "int8"]
LABEL = {"split": "split", "global": "global", "student": "student", "int8": "int8", "moe": "MoE",
         "one_by_one": "1 by 1", "router": "gate + fresh experts",
         "router_given": "gate + workflow experts"}  # fmt: skip
COLOR = {"split": ACCENT[0], "global": ACCENT[3], "student": ACCENT[4], "int8": ACCENT[1], "moe": ACCENT[5],
         "one_by_one": "#6b7280", "router": ACCENT[6], "router_given": ACCENT[2]}  # fmt: skip
GREY = "#d1d5db"


def _np(x):
    return x.detach().cpu().numpy() if hasattr(x, "detach") else np.asarray(x)


def _segments(res: dict) -> list[tuple]:
    """Stages on one x axis: (name, offset, length) in the order they were trained."""
    cfg = res["cfg"]
    out, off = [], 0
    for name, n in (("split", cfg.split_steps), ("router", cfg.router_steps), ("global", cfg.global_steps),
                    ("student", cfg.student_steps)):  # fmt: skip
        out.append((name, off, max(n, 1)))
        off += max(n, 1)
    return out


def _box(ax, x, y, text, color, w=1.9, h=0.62, fc="white", lw=1.6):
    ax.add_patch(FancyBboxPatch((x - w / 2, y - h / 2), w, h, boxstyle="round,pad=0.04", fc=fc, ec=color, lw=lw))
    ax.text(x, y, text, ha="center", va="center", fontsize=10, color="#111827")


def _arrow(ax, a, b, color="#6b7280", lw=1.2):
    ax.annotate("", xy=b, xytext=a, arrowprops=dict(arrowstyle="-|>", color=color, lw=lw, shrinkA=2, shrinkB=2))


def draw_dag(ax, active: int | None) -> None:
    """The workflow: encode_image, encode_text → fuse_{op} (the input function picks one) → classify."""
    ax.clear()
    ax.set_xlim(-0.1, 10.1)
    ax.set_ylim(0.6, 5.4)
    ax.axis("off")
    _box(ax, 0.75, 4.6, "image $x$", "#6b7280", w=1.5)
    _box(ax, 0.75, 1.4, "text $t$", "#6b7280", w=1.5)
    _box(ax, 3.25, 4.6, "encode_image", ACCENT[0], w=2.5)
    _box(ax, 3.25, 1.4, "encode_text", ACCENT[5], w=2.5)
    _box(ax, 3.25, 3.0, "router $G$", "#111827", w=1.6, h=0.5, fc="#f3f4f6")
    for e, name in enumerate(T.OPS):
        y = 4.6 - 1.6 * e
        on = active == e
        _box(ax, 6.35, y, f"fuse_{name}", ACCENT[2] if on else GREY, w=2.3, fc="#ecfdf5" if on else "white",
             lw=2.4 if on else 1.2)  # fmt: skip
        for src in ((4.55, 4.6), (4.55, 1.4)):
            _arrow(ax, src, (5.15, y), color=ACCENT[2] if on else GREY, lw=1.6 if on else 0.8)
        _arrow(ax, (7.55, y), (8.4, 3.0), color=ACCENT[2] if on else GREY, lw=1.6 if on else 0.8)
    _arrow(ax, (1.55, 4.6), (1.95, 4.6))
    _arrow(ax, (1.55, 1.4), (1.95, 1.4))
    _arrow(ax, (3.25, 1.71), (3.25, 2.75), color="#111827")
    _box(ax, 9.2, 3.0, "classify", "#111827", w=1.5)
    ax.text(9.2, 2.25, r"$(d \circ k)\ \mathrm{mod}\ 10$", ha="center", fontsize=10)
    ax.set_title("workflow DAG = top-1 MoE with a fixed router", fontsize=11)


def animate(res: dict, path: Path, n_frames: int = 50, fps: int = 5) -> Path:
    segs = _segments(res)
    total = segs[-1][1] + segs[-1][2]
    ex = res["examples"]
    show = _np(res["show"])
    order = [int(i) for i in show[:: max(1, len(show) // 9)]][:9]  # examples of every operation
    G_fixed = _np(res["G_fixed"])
    M = res["M"]
    perm = res["router_perm"][M]  # fresh experts, seed 1: columns reordered to match the operations best
    gap = np.full((len(G_fixed), 1), np.nan)
    snaps = [np.concatenate([_np(s[0]), gap, _np(s[M])[:, list(perm)]], 1) for s in res["router_snaps"]]

    fig = plt.figure(figsize=(13.0, 7.2))
    gs = fig.add_gridspec(2, 5, height_ratios=[1.0, 1.0], hspace=0.6, wspace=0.55, left=0.06, right=0.99,
                          top=0.94, bottom=0.12)  # fmt: skip
    ax_dag = fig.add_subplot(gs[0, :2])
    ax_ex, ax_gf, ax_gl = (fig.add_subplot(gs[0, i]) for i in (2, 3, 4))
    ax_cur = fig.add_subplot(gs[1, :2])
    ax_acc, ax_size, ax_time = (fig.add_subplot(gs[1, i]) for i in (2, 3, 4))

    # router matrices
    cmap = ListedColormap(["white", ACCENT[2]]).with_extremes(bad="#9ca3af")
    ticks = np.arange(T.E)
    for ax, title in ((ax_gf, "$G$ fixed (workflow)"), (ax_gl, "$G$ learned (gate)")):
        if ax is ax_gf:
            ax.set_xticks(ticks, T.OP_SYMBOL, fontsize=10)
        else:
            ax.set_xticks([0, 1, 2, 4, 5, 6], list(T.OP_SYMBOL) + [f"$e_{q}$" for q in perm], fontsize=10)
        ax.set_xlabel("expert $e$ (index)", fontsize=10)
        ax.set_ylabel("test sample (index)", fontsize=10)
        ax.set_title(title, fontsize=11)
        ax.tick_params(labelsize=10)
    ax_gf.imshow(G_fixed, cmap=cmap, vmin=0, vmax=1, aspect="auto", interpolation="nearest")
    im_gl = ax_gl.imshow(snaps[0], cmap=cmap, vmin=0, vmax=1, aspect="auto", interpolation="nearest")
    rec = _np(res["router_recovery"])

    # training curves
    curves = res["curves"]
    lines = []  # (line, xs, ys)

    def add(xs, ys, color, ls="-", label=None):
        for m in range(ys.shape[1]):
            (ln,) = ax_cur.plot([], [], ls, color=color, lw=1.5, alpha=0.9, label=label if m == 0 else None)
            lines.append((ln, xs, ys[:, m]))

    off = {name: o for name, o, _ in segs}
    sp = curves["split"]
    xs = np.array([c[0] for c in sp]) + off["split"]
    add(xs, np.stack([_np(c[1]) for c in sp]), ACCENT[0], label="image (digit)")
    add(xs, np.stack([_np(c[2]) for c in sp]), ACCENT[5], label="text (number)")
    add(xs, np.stack([_np(c[3]) for c in sp]), ACCENT[2], label="fuse (table)")
    add(xs, np.stack([_np(c[4]) for c in sp]), "#111827", "--", label="chained")
    cv = curves["router"]
    xr, yr = np.array([c[0] for c in cv]) + off["router"], np.stack([_np(c[1]) for c in cv])  # (C, 2M)
    add(xr, yr[:, :M], COLOR["router_given"], label="gate, given experts")
    add(xr, yr[:, M:], COLOR["router"], label="gate, fresh experts")
    for name, color, label in (
        ("global", COLOR["global"], "global"),
        ("student", COLOR["student"], "student"),
    ):
        cv = curves[name]
        add(np.array([c[0] for c in cv]) + off[name], np.stack([_np(c[1]) for c in cv]), color, label=label)
    for name, o, n in segs:
        ax_cur.axvline(o, color=GREY, lw=1)
        ax_cur.text(o + n / 2, 1.03, {"split": "2 split", "router": "1b gate", "global": "3",
                                      "student": "4 distil"}[name], ha="center", fontsize=9)  # fmt: skip
    ax_cur.set_xlim(0, total)
    ax_cur.set_ylim(0, 1.62)
    ax_cur.set_yticks(np.linspace(0, 1, 6))
    ax_cur.set_xlabel("training step, stages in order (steps)", fontsize=10)
    ax_cur.set_ylabel("test accuracy (dimensionless)", fontsize=10)
    ax_cur.tick_params(labelsize=10)
    ax_cur.legend(fontsize=9, loc="upper center", ncol=3, framealpha=0.9, handlelength=1.5, columnspacing=0.8)

    # bars with one dot per seed
    acc = {k: _np(v) for k, v in res["acc"].items()}
    t_ms = {k: _np(v) * 1e3 for k, v in res["time"].items()}
    kb = {k: res["bytes"][k] / 1e3 for k in MODELS}
    lo = min(float(min(v.min() for v in acc.values() if v is not None)), 0.85)
    ready_at = {"split": off["split"] + segs[0][2], "global": off["global"] + segs[2][2],
                "student": total, "int8": total}  # fmt: skip
    time_models = ["one_by_one", "split", "moe", "student", "int8"]
    ready_at["moe"], ready_at["one_by_one"] = ready_at["split"], ready_at["split"]

    def bars(ax, keys, values, ylabel, title, ylim, now, log=False, dots=True):
        ax.clear()
        for i, k in enumerate(keys):
            if now < ready_at[k]:
                continue
            v = np.atleast_1d(values[k])
            ax.bar(i, v.mean(), color=COLOR[k], alpha=0.35, width=0.7)
            if dots:
                ax.plot(np.full(len(v), i) + np.linspace(-0.12, 0.12, len(v)), v, "o", color=COLOR[k], ms=4)
        ax.set_xticks(range(len(keys)), [LABEL[k] for k in keys], rotation=40, ha="right", fontsize=10)
        ax.set_xlim(-0.6, len(keys) - 0.4)
        if log:
            ax.set_yscale("log")
        ax.set_ylim(*ylim)
        ax.set_ylabel(ylabel, fontsize=10)
        ax.set_title(title, fontsize=11)
        ax.tick_params(axis="y", labelsize=10)

    tmax = max(float(v.max()) for v in t_ms.values())
    tmin = min(float(v.min()) for v in t_ms.values())

    def update(f):
        p = min(1.0, f / (0.8 * (n_frames - 1)))
        now = p * total
        for ln, xs_, ys_ in lines:
            keep = xs_ <= now + 1e-9
            ln.set_data(xs_[keep], ys_[keep])
        k = min(len(snaps) - 1, int(np.clip((now - off["router"]) / segs[1][2], 0, 1) * (len(snaps) - 1) + 1e-9))
        im_gl.set_data(snaps[k])
        ax_gl.set_title(f"learned: {rec[0]:.0%} | {rec[M]:.0%} match" if k == len(snaps) - 1 else "$G$ learned (gate)",
                        fontsize=11)  # fmt: skip
        i = order[(f // 5) % len(order)]
        op = int(ex["op"][i])
        draw_dag(ax_dag, op)
        ax_ex.clear()
        ax_ex.imshow(_np(ex["x"][i]).reshape(28, 28), cmap="gray_r", vmin=0, vmax=1)
        ax_ex.axis("off")
        ax_ex.set_title(f"“{T.words(ex['tok'][i])}”", fontsize=11)
        d, kk, y = int(ex["d"][i]), int(ex["k"][i]), int(ex["y"][i])
        ps, pq = int(ex["pred_global"][0, i]), int(ex["pred_int8"][0, i])
        ax_ex.text(0.5, -0.08, f"{d} {T.OP_SYMBOL[op]} {kk} mod 10 = {y}", transform=ax_ex.transAxes, ha="center",
                   va="top", fontsize=10)  # fmt: skip
        ax_ex.text(0.5, -0.22, f"DAG {ps} {'✓' if ps == y else '✗'}   int8 {pq} {'✓' if pq == y else '✗'}",
                   transform=ax_ex.transAxes, ha="center", va="top", fontsize=10)  # fmt: skip
        bars(ax_acc, MODELS, acc, "test accuracy (dimensionless)", "accuracy", (lo - 0.03, 1.0), now)
        bars(ax_size, MODELS, kb, "size (kB)", "weights", (0, max(kb.values()) * 1.15), now, dots=False)
        bars(ax_time, time_models, t_ms, "time per 1000 samples (ms)", "CPU, 1 thread", (tmin / 2, tmax * 2), now,
             log=True)  # fmt: skip
        return []

    update(0)
    return save_gif_shared_palette(fig, update, list(range(n_frames)), path, fps=fps)


def stages_png(res: dict, path: Path) -> Path:
    """Router confusion (true operation × chosen expert) for every seed, and accuracy against bytes."""
    conf = _np(res["router_confusion"])
    M = res["M"]
    panels = [0] + list(range(M, 2 * M))  # gate + workflow experts (seed 1), gate + fresh experts (every seed)
    n = len(panels)
    fig, axes = plt.subplots(1, n + 1, figsize=(4.3 * (n + 1), 4.4), gridspec_kw={"width_ratios": [1] * n + [1.4]})
    for j, m in enumerate(panels):
        ax = axes[j]
        im = ax.imshow(conf[m], cmap="Greens", vmin=0, vmax=1 / T.E)
        for a in range(T.E):
            for b in range(T.E):
                c = "white" if conf[m, a, b] > 0.2 else "#111827"
                ax.text(b, a, f"{conf[m, a, b]:.2f}", ha="center", va="center", fontsize=10, color=c)
        ax.grid(False)
        ax.set_xticks(range(T.E), [f"$e_{q}$" for q in range(T.E)], fontsize=10)
        ax.set_yticks(range(T.E), T.OPS, fontsize=10)
        ax.set_xlabel("expert chosen by the gate (index)", fontsize=10)
        ax.set_ylabel("operation in the text (category)", fontsize=10)
        kind = "workflow experts" if m < M else "fresh experts"
        ax.set_title(f"gate + {kind}, seed {m % M + 1}\n{float(res['router_recovery'][m]):.0%} match", fontsize=11)
        cb = fig.colorbar(im, ax=ax, fraction=0.046)
        cb.set_label("fraction of test samples (dimensionless)", fontsize=10)
        cb.ax.tick_params(labelsize=10)
    ax = axes[-1]
    for k in MODELS + ["router_given", "router"]:
        b = res["bytes"].get(k, res["bytes"]["split"]) / 1e3
        a = _np(res["acc"][k])
        ax.plot(np.full(len(a), b), a, "o", color=COLOR[k], ms=7, label=LABEL[k])
    allb = [v / 1e3 for v in res["bytes"].values()]
    ax.set_xlim(0, max(allb) * 1.1)
    alla = np.concatenate([_np(v) for v in res["acc"].values()])
    ax.set_ylim(max(0.0, alla.min() - 0.05), 1.0)
    ax.set_xlabel("stored weights (kB)", fontsize=10)
    ax.set_ylabel("test accuracy (dimensionless)", fontsize=10)
    ax.set_title("accuracy vs size (one dot per seed)", fontsize=11)
    ax.tick_params(labelsize=10)
    ax.legend(fontsize=9, loc="lower left")
    fig.tight_layout()
    return save_png(fig, path)


def write_tables(res: dict, out_dir: Path) -> None:
    M = res["M"]
    with open(out_dir / "stages.csv", "w", newline="", encoding="utf-8") as f:
        wr = csv.writer(f)
        wr.writerow(["stage", "seed", "accuracy", "params", "bytes", "time_ms_per_1000"])
        for k in ["split", "router_given", "router", "global", "student", "int8"]:
            for m in range(M):
                t = res["time"].get(k)
                wr.writerow([k, m + 1, f"{float(res['acc'][k][m]):.4f}", res["params"].get(k, res["params"]["split"]),
                             res["bytes"].get(k, res["bytes"]["split"]),
                             f"{float(t[m]) * 1e3:.3f}" if t is not None else ""])  # fmt: skip
    eqv = res["equivalence"]
    with open(out_dir / "equivalence.csv", "w", newline="", encoding="utf-8") as f:
        wr = csv.writer(f)
        wr.writerow(["quantity", "value"])
        for dt in ("float64", "float32"):
            for key, v in eqv[dt].items():
                wr.writerow([f"{dt}_{key}", v])
        for rule, (need, n) in eqv["jobs"].items():
            wr.writerow([f"jobs_needed_{rule}", f"{need} of {n}"])
        for k in ("split", "moe", "one_by_one"):
            for m in range(M):
                wr.writerow([f"time_ms_per_1000_{k}_seed{m + 1}", f"{float(res['time'][k][m]) * 1e3:.3f}"])
        for m in range(M):
            wr.writerow([f"router_given_experts_identity_match_seed{m + 1}", f"{float(res['router_identity'][m]):.4f}"])
            wr.writerow([f"router_fresh_experts_best_match_seed{m + 1}", f"{float(res['router_recovery'][M + m]):.4f}"])


## Run

Change the parameters (or any file above) and run again.

In [ ]:
import importlib
from pathlib import Path

from IPython.display import Image, display

from problems._common.runtime import load_meta, resolve_params

params = {
    'seeds': 2,
    'split_steps': 1000,
    'router_steps': 800,
    'global_steps': 400,
    'student_steps': 3000,
    'student_hidden': 64,
    'temperature': 4.0,
    'balance_weight': 0.01
}

# Re-import so edits in the %%writefile cells above take effect.
for name in [m for m in sys.modules if m.startswith("problems.ai.moe_dag_pipeline")]:
    del sys.modules[name]
main = importlib.import_module("problems.ai.moe_dag_pipeline.main")
problem_dir = Path("problems/ai/moe_dag_pipeline")
output = main.run(resolve_params(load_meta(problem_dir), params), Path("out"))
display(Image(filename=str(output)))
